# SPATIAL INTELLIGENCE - PART 1

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\Win11\Desktop\AIA2026\GraphLM Seminar\GraphML_RaniaChihaoui-main\GraphML_RaniaChihaoui\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Import the OBJ file

In [ ]:
# Import all OBJ files
building_objects = Topology.ByOBJPath("C:\\Users\\Win11\\Desktop\\AIA2026\\GraphLM Seminar\\GraphML_RaniaChihaoui-main\\Assigment-1_RaniaChihaoui\\assets\\obj")
door_objects = Topology.ByOBJPath("C:\Users\Win11\Desktop\AIA2026\GraphLM Seminar\GraphML_RaniaChihaoui-main\Assigment-1_RaniaChihaoui\assets\obj")
window_objects = Topology.ByOBJPath("C:\Users\Win11\Desktop\AIA2026\GraphLM Seminar\GraphML_RaniaChihaoui-main\Assigment-1_RaniaChihaoui\assets\obj")

# Build clusters
building_cluster = Cluster.ByTopologies(building_objects)
door_cluster_raw = Cluster.ByTopologies(door_objects)
window_cluster_raw = Cluster.ByTopologies(window_objects)

# Get faces
faces = Topology.Faces(building_cluster)
door_faces = Topology.Faces(door_cluster_raw)
window_faces = Topology.Faces(window_cluster_raw)

print("Building faces:", len(faces))
print("Door faces:", len(door_faces))
print("Window faces:", len(window_faces))

## 6. Show the geometry

In [ ]:
Topology.Show(building_objects,
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Build cellcomplex

In [ ]:
cellcomplex = CellComplex.ByFaces(faces, tolerance=0.001)
cells = Topology.Cells(cellcomplex)
cell_vertices = [Topology.Centroid(c) for c in cells]
print("Cells:", len(cells))

## 8. Add apertures

In [ ]:
cc_building = Topology.AddApertures(cellcomplex,
                                     door_faces + window_faces,
                                     exclusive=False,
                                     subTopologyType="face",
                                     tolerance=0.5)
print("Apertures added!")

## 9. Label cells to identify rooms

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

for i, v in enumerate(cell_vertices):
    fig.add_trace(go.Scatter3d(
        x=[v.X()], y=[v.Y()], z=[v.Z()],
        mode='text',
        text=[str(i)],
        textfont=dict(size=16, color='red'),
        showlegend=False
    ))

for e in Topology.Edges(cellcomplex):
    vs = Topology.Vertices(e)
    fig.add_trace(go.Scatter3d(
        x=[vs[0].X(), vs[1].X()],
        y=[vs[0].Y(), vs[1].Y()],
        z=[vs[0].Z(), vs[1].Z()],
        mode='lines',
        line=dict(color='black', width=1),
        showlegend=False
    ))

fig.update_layout(
    width=900, height=700,
    showlegend=False,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor='white',
        camera=dict(
            eye=dict(x=-1.5, y=-1.5, z=1.2)
        )
    ),
    paper_bgcolor='white'
)
fig.show()

## 10. Privacy level assignment

In [ ]:
# Privacy level assignment
# Level 1 = Public, Level 2 = Semi-Private, Level 3 = Private

room_info = {
    7:  {"name": "Entrance Lobby",  "floor": "ground", "privacy": 1},
    5:  {"name": "TV Room",         "floor": "ground", "privacy": 2},
    3:  {"name": "Guest Bedroom",   "floor": "ground", "privacy": 3},
    11: {"name": "Kitchen",         "floor": "ground", "privacy": 2},
    4:  {"name": "Stairs",          "floor": "both",   "privacy": 1},
    13: {"name": "Dining",          "floor": "ground", "privacy": 2},
    12: {"name": "Living Room",     "floor": "ground", "privacy": 1},
    2:  {"name": "Corridor",        "floor": "first",  "privacy": 1},
    8:  {"name": "Office",          "floor": "first",  "privacy": 2},
    0:  {"name": "Bedroom-1",       "floor": "first",  "privacy": 3},
    1:  {"name": "Bedroom-2",       "floor": "first",  "privacy": 3},
    9:  {"name": "Bedroom-3",       "floor": "first",  "privacy": 3},
    10: {"name": "Bedroom-4",       "floor": "first",  "privacy": 3},
    6:  {"name": "Family Room",     "floor": "first",  "privacy": 2},
}

# Color mapping
privacy_colors = {1: "red", 2: "orange", 3: "blue"}

print("Room info assigned!")
for k, v in room_info.items():
    print(f"Cell {k}: {v['name']} - Privacy Level {v['privacy']}")

## 11. Assign colors and labels to cell vertices

In [ ]:
for i, v in enumerate(cell_vertices):
    if i in room_info:
        privacy = room_info[i]["privacy"]
        name = room_info[i]["name"]
        color = privacy_colors[privacy]
        d = Dictionary.ByKeysValues(
            ["size", "color", "name", "privacy"],
            [8, color, name, privacy]
        )
        Topology.SetDictionary(v, d)

print("Colors assigned!")

## 12. Build the adjacency graph using door apertures

In [ ]:
# Build graph with door waypoints
door_apt_centroids = []
cell_edges_privacy = []

for i in range(len(cells)):
    for j in range(i+1, len(cells)):
        faces_i = Topology.Faces(cells[i])
        faces_j = Topology.Faces(cells[j])
        for fi in faces_i:
            for fj in faces_j:
                if Topology.IsSame(fi, fj):
                    apts = Topology.Apertures(fi)
                    if apts and len(apts) > 0:
                        apt_top = Topology.ApertureTopologies(apts[0])
                        apt_centroid = Topology.Centroid(apts[0])
                        is_door = any(Face.IsCoplanar(
                            Topology.ApertureTopologies(apts[0])[0] if apt_top else apts[0],
                            df, tolerance=0.5) for df in door_faces)
                        if is_door:
                            door_apt_centroids.append(apt_centroid)
                            e1 = Edge.ByVertices([cell_vertices[i], apt_centroid])
                            e2 = Edge.ByVertices([apt_centroid, cell_vertices[j]])
                            cell_edges_privacy.extend([e1, e2])
                        else:
                            e = Edge.ByVertices([cell_vertices[i], cell_vertices[j]])
                            cell_edges_privacy.append(e)
                    break

# Add stair + entrance connections manually
e_stair1 = Edge.ByVertices([cell_vertices[4], cell_vertices[2]])
e_stair2 = Edge.ByVertices([cell_vertices[4], cell_vertices[7]])
entrance_centroid = Topology.Centroid(door_faces[1])
e_entrance = Edge.ByVertices([entrance_centroid, cell_vertices[7]])
door_apt_centroids.append(entrance_centroid)
cell_edges_privacy.extend([e_stair1, e_stair2, e_entrance])

all_verts = cell_vertices + door_apt_centroids
g_privacy = Graph.ByVerticesEdges(all_verts, cell_edges_privacy)
print("Vertices:", len(Graph.Vertices(g_privacy)))
print("Edges:", len(Graph.Edges(g_privacy)))

## 13. Visualize

In [ ]:
# Set all face colors and opacities
for f in Topology.Faces(cc_building):
    d = Dictionary.ByKeysValues(["color", "opacity"], ["white", 0.05])
    Topology.SetDictionary(f, d)

for f in door_faces:
    d = Dictionary.ByKeysValues(["color", "opacity"], ["brown", 0.6])
    Topology.SetDictionary(f, d)

for f in window_faces:
    d = Dictionary.ByKeysValues(["color", "opacity"], ["lightblue", 0.6])
    Topology.SetDictionary(f, d)

for v in Topology.Vertices(cc_building):
    d = Dictionary.ByKeysValues(["size", "color"], [0, "white"])
    Topology.SetDictionary(v, d)

for i, v in enumerate(cell_vertices):
    if i in room_info:
        color = privacy_colors[room_info[i]["privacy"]]
        d = Dictionary.ByKeysValues(["size", "color"], [15, color])
        Topology.SetDictionary(v, d)

for v in door_apt_centroids:
    d = Dictionary.ByKeysValues(["size", "color"], [8, "gray"])
    Topology.SetDictionary(v, d)

door_cluster = Cluster.ByTopologies(door_faces)
window_cluster = Cluster.ByTopologies(window_faces)

Topology.Show(cc_building, door_cluster, window_cluster, g_privacy,
              vertexSizeKey="size",
              vertexColorKey="color",
              faceColorKey="color",
              opacityKey="FakeKey",
              faceOpacity=0.1,
              edgeColor="gray",
              edgeWidth=2,
              width=800,
              height=600,
              backgroundColor="white",
              renderer=renderer)